____
### 0. Preamble
This notebook serves to explain the data processing methods for creating the dataset used to train the autoregression model for the SQL suggestor

____
### 1. Imports

In [1]:
from problembank import PROBLEM_BANK, BLOCK_TYPES
from torch.nn.utils.rnn import pad_sequence
import random
import torch
import itertools
from collections import defaultdict

- the extensive problem bank generated from the previous reinforcement learning approach is abstracted into the file `problembank.py` and imported wholesale here for ease

____
### 2. Converting data structure of problems
- in the original `PROBLEM_BANK`, each problem is a tree built for the RL environment's stack/Frame mechanics
- in the autoregression problem, the problems need to be a flat continuous token sequence for training a autoregressive model
- therefore the questions need to be flattened into a singular array list, which is done by the function `flatten(tree)`


In [2]:
def flatten(tree):
    if tree["subqueries"] == {}: #base case with no extra node
        return tree["blocks"]
    else:
        out = [] #empty array 
        blocks = tree["blocks"] #get blocks
        subqueries = tree.get("subqueries", {}) #get node
        for i, tok in enumerate(blocks):
            out.append(tok)
            if tok == "SUBQUERY_START":
                out.extend(flatten(subqueries[i]))
        return out

confirmation that flattening works:

In [3]:
nested = next(p for p in PROBLEM_BANK if p.get("subqueries"))
print(f"original:{nested}" )
print(f"flattened:{flatten(nested)}")

original:{'blocks': ['SELECT', 'COLUMN', 'FROM', 'TABLE', 'WHERE', 'COLUMN', 'IN', 'SUBQUERY_START', 'SUBQUERY_END'], 'subqueries': {7: {'blocks': ['SELECT', 'AGG_FUNC', 'STAR', 'FROM', 'TABLE', 'WHERE', 'COLUMN', 'IS_NOT_NULL'], 'subqueries': {}}}}
flattened:['SELECT', 'COLUMN', 'FROM', 'TABLE', 'WHERE', 'COLUMN', 'IN', 'SUBQUERY_START', 'SELECT', 'AGG_FUNC', 'STAR', 'FROM', 'TABLE', 'WHERE', 'COLUMN', 'IS_NOT_NULL', 'SUBQUERY_END']


____
### 3. Vocabulary 
- in the rl representation, the blocks available `BLOCK_TYPES` represent the tokens in the sequence

In [4]:
print(BLOCK_TYPES)

['SELECT', 'FROM', 'WHERE', 'GROUP_BY', 'HAVING', 'ORDER_BY', 'LIMIT', 'JOIN', 'DISTINCT', 'COLUMN', 'STAR', 'TABLE', 'IS_NULL', 'IS_NOT_NULL', 'OPERATOR', 'LIKE', 'IN', 'NOT', 'AND', 'OR', 'VALUE', 'AGG_FUNC', 'ASC', 'DESC', 'OFFSET', 'ON', 'SUBQUERY_START', 'SUBQUERY_END']


- for extra bookkeeping, 3 more tokens need to be added to `BLOCK_TYPES` for bookkeeping purposes
    1. `<PAD>` : padding used to make batched sequences the same length (a "no-token" placeholder)
    2. `<SOS>` : start of sequence, to mark the beginning of the sequence
    3. `<EOS>` : end of sequence, to mark the end of the sequence
- these 3 tokens are used to standardize sequence boundaries (`<SOS>`/`<EOS>`) and enable padding (`<PAD>`) during batching; raw sequences can still have different lengths.

In [5]:
SPECIALS = ["<PAD>", "<SOS>", "<EOS>"]
TOKENS = SPECIALS + list(BLOCK_TYPES)
N_TOKENS = len(TOKENS)
print(f"Number of tokens: {N_TOKENS} types")
print(TOKENS)

Number of tokens: 31 types
['<PAD>', '<SOS>', '<EOS>', 'SELECT', 'FROM', 'WHERE', 'GROUP_BY', 'HAVING', 'ORDER_BY', 'LIMIT', 'JOIN', 'DISTINCT', 'COLUMN', 'STAR', 'TABLE', 'IS_NULL', 'IS_NOT_NULL', 'OPERATOR', 'LIKE', 'IN', 'NOT', 'AND', 'OR', 'VALUE', 'AGG_FUNC', 'ASC', 'DESC', 'OFFSET', 'ON', 'SUBQUERY_START', 'SUBQUERY_END']


- dictionaries need to be initialised to easily convert back and forth between an `IDX` and the corresponding `TOKEN`

In [6]:
TOKEN_TO_ID = {tok: i for i, tok in enumerate(TOKENS)}
ID_TO_TOKEN = {i: tok for tok, i in TOKEN_TO_ID.items()}
PAD_ID, SOS_ID, EOS_ID = TOKEN_TO_ID["<PAD>"], TOKEN_TO_ID["<SOS>"], TOKEN_TO_ID["<EOS>"]

____
### 4. Constructing dataset
- the following process is conducted to every problem type inside `PROBLEM_BANK`
1. flatten the problem
    - converts the tree representation into a linear sequence of tokens
2. convert linear sequence of tokens into a tensor of the corresponding index
    - adds a `<SOS>` token to the start and a `<EOS>` token to the end before converting it to a tensor of corresponding index
- these 2 steps are done by `normalise_data(problem)`

In [48]:
def normalise_data(problem):
    flat = flatten(problem)
    ids = [SOS_ID] + [TOKEN_TO_ID[t] for t in flat] + [EOS_ID]
    return torch.tensor(ids, dtype=torch.long)

def to_token_seq(bank):
    sequences = []
    for problem in bank:
        sequences.append(normalise_data(problem))
    return sequences

In [49]:
tensorlst = to_token_seq(PROBLEM_BANK)
lengths = [len(s) for s in tensorlst]
print(f"{len(tensorlst)} sequences | min len {min(lengths)}, max len {max(lengths)}, mean len {sum(lengths)/len(lengths):.1f}")

3020 sequences | min len 6, max len 38, mean len 20.3


after normalising the data and storing it in a list of tensors `tensorlst`, the data is stored inside a `Dataset` object 
- a `Dataset` object is needed to use inside a `DataLoader` later on

In [9]:
class SQLSequenceDataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx]

- after wrapping the list of tensors in a `Dataset` object, it is split randomly into a training set and validation set

In [10]:
dataset = SQLSequenceDataset(tensorlst)
n_val = max(1, int(0.05 * len(dataset))) #takes 5% of the number of rows of data, or 1, whichever is higher
train_set, val_set = torch.utils.data.random_split(dataset, [len(dataset) - n_val, n_val])
#5% of the data reserved as the validation set
print(f"train/val split: {len(train_set)}/{len(val_set)}")

train/val split: 2869/151


- a `DataLoader` object is then created, taking in the training set and a `collate_fn` 
- iterating through the `DataLoader` will then return batches of data from the training set
- the `collate_fn` taken as input will then pad and format all the data to form the teacher forcing pair

In [11]:
def collate(batch):
    padded = pad_sequence(batch, batch_first=True, padding_value=PAD_ID) #pads the value 
    input_ids = padded[:, :-1] #select up until the second last item, padding the rest
    target_ids = padded[:, 1:] #select after the first item, until the last item, dropping the first item as the window sliced remains the same size
    return input_ids, target_ids

In [12]:
loader = torch.utils.data.DataLoader(train_set, batch_size=8, shuffle=True, collate_fn=collate)
input_ids, target_ids = next(iter(loader))
print(f"One training batch: input_ids {tuple(input_ids.shape)}, "
          f"target_ids {tuple(target_ids.shape)}")

One training batch: input_ids (8, 27), target_ids (8, 27)


The functions in this section are defined directly in this notebook (you can optionally move them into a separate module like `dataprep.py` later).

____
### 5. Deliberate splitting of dataset
- the goal here is not just a random train/val split, but to *deliberately* create evaluation sets that test generalization to unseen **variants** of specific SQL clauses ("knobs").
- we do this by holding out certain knob variants while ensuring their tokens still appear elsewhere in training (so we are testing recombination/generalization, not unseen vocabulary).

#### 5.1 Scaling the problem bank
- the original `PROBLEM_BANK` has 3,020 problems; we build an expanded `PROBLEM_BANK2` with 5,020 problems so we can (a) keep an in-distribution train/val split and (b) reserve additional held-out (OOD) sets for generalization tests.
- the original environment used 13 hand-written SQL query templates which is too small to teach a model as it could just memorise a lookup table instead
- `PROBLEM_BANK2` is generated by a combinatorial generator with six "knobs" (SELECT/JOIN/WHERE/GROUPBY/ORDERBY/LIMIT).
    - each knob represents a chunk of the SQL clause with a list of ways to write the clause (or skip it entirely).
- the problem bank is increased through adding new `GROUP_BY` variants
    - flat queries: 2,880 → 4,800
    - level 1 nesting: 120 → 200
    - level 2 nesting: no change

In [ ]:
#VARIANTS
SELECT_ITEMS = [
    ["COLUMN"],
    ["DISTINCT", "COLUMN"],
    ["STAR"],
    ["AGG_FUNC", "COLUMN"],
    ["AGG_FUNC", "STAR"],
]
 
JOINS = [
    None,
    ["JOIN", "TABLE", "ON", "COLUMN", "OPERATOR", "COLUMN"],
]

WHERE_VARIANTS = [
    None,
    ["WHERE", "COLUMN", "OPERATOR", "VALUE"],
    ["WHERE", "COLUMN", "IS_NULL"],
    ["WHERE", "COLUMN", "IS_NOT_NULL"],
    ["WHERE", "COLUMN", "LIKE", "VALUE"],
    ["WHERE", "NOT", "COLUMN", "LIKE", "VALUE"],
    ["WHERE", "COLUMN", "OPERATOR", "VALUE", "AND", "COLUMN", "OPERATOR", "VALUE"],
    ["WHERE", "COLUMN", "OPERATOR", "VALUE", "OR", "COLUMN", "OPERATOR", "VALUE"],
]
 
GROUPBY_VARIANTS = [
    None,
    ["GROUP_BY", "COLUMN"],
    ["GROUP_BY", "COLUMN", "HAVING", "AGG_FUNC", "COLUMN", "OPERATOR", "VALUE"],
    ["GROUP_BY", "COLUMN", "HAVING", "AGG_FUNC", "COLUMN", "OPERATOR", "VALUE",
     "AND", "AGG_FUNC", "COLUMN", "OPERATOR", "VALUE"],
    ["GROUP_BY", "COLUMN", "HAVING", "AGG_FUNC", "COLUMN", "OPERATOR", "VALUE",
     "OR", "AGG_FUNC", "COLUMN", "OPERATOR", "VALUE"],
]
 
ORDERBY_VARIANTS = [
    None,
    ["ORDER_BY", "COLUMN"],
    ["ORDER_BY", "COLUMN", "ASC"],
    ["ORDER_BY", "COLUMN", "DESC"],
]
 
LIMIT_VARIANTS = [
    None,
    ["LIMIT"],
    ["LIMIT", "OFFSET"],
]
 

- every legal combination of these 6 knobs produce 4,800 flat queries (linear with no subqueries)
- a simple subquery pool is also created, containing 30 subqueries
- a second pass introduced subqueries by subbing a subquery into `WHERE ... IN` patterns, producing around ~200 problems with 1 level of nesting.
- widening the subquery generation loop to sample the full range of `ORDERBY_VARIANTS` and `LIMIT_VARIANTS` also increases the problem bank
- a third pass nested 1 more subqueries within the 1 level nesting to create 20 problem with 2 levels of nesting 

this final problem set means the generator has 27 independent choice-values across 6 knobs (5 select × 2 join × 8 where × 5 groupby × 4 orderby × 3 limit)
- models would thus be unable to memorise a simple lookup table and would have to actually learn the patterns

In [27]:
from problembank2 import (PROBLEM_BANK as PROBLEM_BANK2,
                          SIMPLE_SUBQUERY_POOL)
from problembank2 import (assemble, make_problem)

print(f"Total number of problems: {len(PROBLEM_BANK2)} problems")

Total number of problems: 5020 problems


#### 5.2 Splitting of `PROBLEM_BANK2`
- `PROBLEM_BANK2` (5,020 problems) is split into:
  - an **in-distribution** training set and a small **in-distribution** validation set (random 5% split of the non-held-out problems), and
  - separate **OOD evaluation sets** where one knob (or multiple knobs) take held-out variant values.
1. Holding out variants for each knob
    - A variant is and can only be held out if the tokens it introduces are also seen elsewhere in training
    - if a certain variant is the SOLE SOURCE of a token, it cannot be held out (that would test the model on vocabulary it never saw during training, not generalization).

In [22]:
KNOBS = {
    "select": SELECT_ITEMS,
    "join": JOINS,
    "where": WHERE_VARIANTS,
    "groupby": GROUPBY_VARIANTS,
    "orderby": ORDERBY_VARIANTS,
    "limit": LIMIT_VARIANTS,
}

- based on observation of the variants, the "join" knob cannot be held out as there is only 1 variant exclusive of the `None` version

In [21]:
HELD_OUT = {
    "select": {4},     # ["AGG_FUNC", "STAR"]
    "where": {6, 7},   # AND-chained, OR-chained conditions
    "groupby": {1},    # bare "GROUP_BY COLUMN", no HAVING
    "orderby": {1},    # bare "ORDER_BY COLUMN", no ASC/DESC
    "limit": {1},      # bare "LIMIT", no OFFSET
}

- all held out variants for each knob are then double checked to see if they are safe 

In [ ]:
def audit_token_sources():
    token_sources = defaultdict(set)
    for kname, variants in KNOBS.items(): #iterate through all variants and knobs
        for i, v in enumerate(variants): 
            if not v:
                continue
            for tok in set(v):
                token_sources[tok].add((kname, i))
    all_safe = True
    for kname, held in HELD_OUT.items():
        for idx in held:
            v = KNOBS[kname][idx]
            for tok in set(v):
                other_sources = token_sources[tok] - {(kname, idx)}
                if not other_sources:
                    all_safe = False
                    raise AssertionError(
                        f"HELD_OUT['{kname}']={idx} ({v}) is the SOLE source of "
                        f"token {tok!r} -- holding it out would zero-expose it, "
                        f"not test generalization.")
    if all_safe:
        print("Audit OK: all held-out variants are safe (no token is solely sourced by a held-out variant).")

audit_token_sources()

Audit OK: all held-out variants are safe (no token is solely sourced by a held-out variant).


2. Tagging the problems 
    - all the problems are tagged before they can be held out for easier bookkeeping
    - the tagging stores each problem as a tuple in the format `(problem, tags)`

In [29]:
def generate_flat_with_tags():
    tagged = []
    for si, select_item in enumerate(SELECT_ITEMS):
        for ji, join in enumerate(JOINS):
            for wi, where in enumerate(WHERE_VARIANTS):
                for gi, groupby in enumerate(GROUPBY_VARIANTS):
                    for oi, orderby in enumerate(ORDERBY_VARIANTS):
                        for li, limit in enumerate(LIMIT_VARIANTS):
                            blocks = assemble(select_item, join, where, groupby, orderby, limit)
                            problem = make_problem(blocks)
                            tags = {"select": si, "join": ji, "where": wi,
                                    "groupby": gi, "orderby": oi, "limit": li}
                            tagged.append((problem, tags))
    return tagged

- the tag of each problem stores the metadata of the problem in the form of a dictionary:
    ```python
    tags = {"select": si, "join": ji, "where": wi, "groupby": gi, "orderby": oi, "limit": li}
    ```
- each key's value stores the index of the corresponding variant used to create the problem 

In [30]:
def generate_subquery_with_tags():
    tagged = []
    rng = random.Random(0)
    for si, select_item in enumerate(SELECT_ITEMS):
        for ji, join in enumerate(JOINS):
            for gi, groupby in enumerate(GROUPBY_VARIANTS):
                for oi, orderby in enumerate(ORDERBY_VARIANTS):
                    for li, limit in enumerate(LIMIT_VARIANTS):
                        where = ["WHERE", "COLUMN", "IN", "SUBQUERY_START", "SUBQUERY_END"]
                        blocks = assemble(select_item, join, where, groupby, orderby, limit)
                        pos = blocks.index("SUBQUERY_START")
                        nested = rng.choice(SIMPLE_SUBQUERY_POOL)
                        problem = make_problem(blocks, subqueries={pos: nested})
                        tags = {"select": si, "join": ji, "where": "subquery",
                                "groupby": gi, "orderby": oi, "limit": li}
                        tagged.append((problem, tags))
    return tagged

- the same tagging process is done to problems with subqueries
- however, the value for the key `"where"` is fixed to `"subquery"`

verification that all is tagged(except level 2 nesting):

In [31]:
flat_tagged = generate_flat_with_tags()
subquery_tagged = generate_subquery_with_tags()
all_tagged = flat_tagged + subquery_tagged
print(f"tagged problems: {len(all_tagged)} tagged")

tagged problems: 5400 tagged


helper functions that read the metadata from tagging are also required

In [34]:
def is_held_out(tags, knob):
    val = tags[knob]
    return isinstance(val, int) and val in HELD_OUT.get(knob, set())

- reads the metadata dict `tags` and a specific knob that is checked `knob`
- checks if the the metadata `tags` contains any of the variants of that one knob that is held out, `val in HELD_OUT.get(knob, set())`

In [35]:
def any_held_out(tags):
    return any(is_held_out(tags, k) for k in HELD_OUT)

- reads the metadata dict `tags` and check if any of its variants contain a variant that is held out

In [36]:
def only_this_axis_held_out(tags, knob):
    if not is_held_out(tags, knob):
        return False
    return all(is_held_out(tags, k) is False for k in HELD_OUT if k != knob)

- returns `True` if only the `knob` given is at a held out value and everything else is at an in-distribution value

3. Generating the split between Training set and validation set

In [38]:
def build_multi_knob_split(validation_fract=0.05, seed=0):
    tagged = generate_flat_with_tags() + generate_subquery_with_tags() #tags all
    train_tagged = [(p, t) for p, t in tagged if not any_held_out(t)] # everything with no held out 
    ood_by_knob = {} 
    for knob in HELD_OUT: #everything held out 
        ood_by_knob[knob] = [p for p, t in tagged if only_this_axis_held_out(t, knob)]
    n_held_axes = lambda t: sum(is_held_out(t, k) for k in HELD_OUT)
    ood_compound = [p for p, t in tagged if n_held_axes(t) >= 2] # everything with 2 or more knobs held out
    rng = random.Random(seed)
    train_problems = [p for p, t in train_tagged]
    rng.shuffle(train_problems)
    n_val = max(1, int(validation_fract * len(train_problems)))
    val_set = train_problems[:n_val]
    train_set = train_problems[n_val:]
    return train_set, val_set, ood_by_knob, ood_compound, train_tagged

In [40]:
training_set, validation_set, ood_by_knob, ood_compound, train_tagged_all = build_multi_knob_split()
print(f"train:{len(training_set)} problems")
print(f"interp_val: {len(validation_set)} problems")
for knob, probs in ood_by_knob.items():
    print(f"ood[{knob}]:  {len(probs)} problems (only '{knob}' is novel)")
print(f"ood_compound: {len(ood_compound)} problems (2+ knobs novel at once)\n")

train:1277 problems
interp_val: 67 problems
ood[select]:  336 problems (only 'select' is novel)
ood[where]:  384 problems (only 'where' is novel)
ood[groupby]:  336 problems (only 'groupby' is novel)
ood[orderby]:  448 problems (only 'orderby' is novel)
ood[limit]:  672 problems (only 'limit' is novel)
ood_compound: 1880 problems (2+ knobs novel at once)



- `ood_by_knob` is a dictionary with the name of each knob as the key and the list of problems with only that knob held out as the value
- `ood_compound` is a list of problems that have more than 1 knob held out, used as a harder multi-factor generalization test

4. Verify equal distribution of each knob inside training set and validation set

In [41]:
def verify_balance(train_tagged):
    counts = {k: defaultdict(int) for k in KNOBS}
    for p, t in train_tagged:
        for k, v in t.items():
            if isinstance(v, int):
                counts[k][v] += 1
    for k, c in counts.items():
        vals = list(c.values())
        balanced = len(set(vals)) == 1
        print(f"  {k}: counts per in-distribution value = {dict(c)}  "
              f"{'(balanced)' if balanced else '(NOT balanced!)'}")
        
verify_balance(train_tagged_all)

  select: counts per in-distribution value = {0: 336, 1: 336, 2: 336, 3: 336}  (balanced)
  join: counts per in-distribution value = {0: 672, 1: 672}  (balanced)
  where: counts per in-distribution value = {0: 192, 1: 192, 2: 192, 3: 192, 4: 192, 5: 192}  (balanced)
  groupby: counts per in-distribution value = {0: 336, 2: 336, 3: 336, 4: 336}  (balanced)
  orderby: counts per in-distribution value = {0: 448, 2: 448, 3: 448}  (balanced)
  limit: counts per in-distribution value = {0: 672, 2: 672}  (balanced)


5. Process the data
- split is then processed into pytorch tensors that can be used to create the `Dataset` object

In [50]:
training_tensors = to_token_seq(training_set)
validation_tensors = to_token_seq(validation_set)

the final processed tensors (`training_tensors`, `validation_tensors`) are created at the end of this notebook (you can optionally save them out from here in a separate script later).